# Module 04.13 — Cases

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.13 Cases (`cases`, `cases-comments`, `cases-user-actions`)

Kibana Cases provides a **lightweight ticketing and investigation workflow** built on top
of saved objects. It is used by the Security and Observability solutions to track incidents,
group related alerts, and maintain an audit trail of the investigation.

Every case generates three linked saved object types, each with a distinct role:

| Type | Description |
|------|-------------|
| `cases` | The case itself — title, description, status (`open`/`in-progress`/`closed`), severity, assignees, tags, and the external connector reference |
| `cases-comments` | Each comment or alert attachment. Comments are `user` type; alert attachments are `alert` type and carry a reference to the alert ID |
| `cases-user-actions` | An immutable audit log — one record per change to the case (status change, comment added, assignee updated, etc.) |

The three types are linked by `cases-comments` and `cases-user-actions` each holding a
reference to their parent `cases` object. This means restore order matters: the parent
`cases` object must be restored before its comments and user-action records. The
feature-state restore handles this automatically.

One gotcha: if a case references an external connector (e.g. Jira or ServiceNow) via the
`connector` field, and that connector's secrets are unreadable after a cross-cluster
restore (see section 4.12), the external sync for that case will stop working — but the
case data itself is fully intact.

In [ ]:
heading('4.13 Cases — create via API')

case_resp = kibana_post('/api/cases', {
    'title': 'Training Investigation Case',
    'description': 'Investigating unusual order volume spike',
    'tags': ['training', 'ecommerce'],
    'settings': {'syncAlerts': False},
    'severity': 'low',
    'connector': {'id': 'none', 'name': 'none', 'type': '.none', 'fields': None},
})
case_id = case_resp['id']
success(f'Case created: id={case_id}')
pp(case_resp, 'case object')

In [ ]:
# Add a comment
comment_resp = kibana_post(f'/api/cases/{case_id}/comments', {
    'type': 'user',
    'comment': 'Checked the ecommerce index — orders look anomalous between 2024-01-15 and 2024-01-17.',
    'owner': 'cases',
})
comment_id = comment_resp['id']
success(f'Comment added: id={comment_id}')

# Show the three related saved objects
for so_type in ['cases', 'cases-comments', 'cases-user-actions']:
    objs = find_saved_objects(so_type)
    console.print(f'  {so_type}: {len(objs)} object(s)')

info('All three types are included in the kibana feature state snapshot.')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/cases', 'Cases — open the case we just created')
if 'case_id' in dir():
    kibana_link(f'/app/cases/{case_id}', f'Case detail: Training Investigation Case')

In [ ]:
heading('Case — snapshot → delete → restore')

import requests as req

# Ensure case exists (re-runnable)
cases_resp = kibana_get('/api/cases/_find?perPage=50')
existing_case = next((c for c in cases_resp.get('cases', []) if c.get('id') == case_id), None)
if not existing_case:
    warn('Case missing — recreating')
    case_resp = kibana_post('/api/cases', {
        'title': 'Training Investigation Case',
        'description': 'Investigating unusual order patterns in the eCommerce dataset.',
        'tags': ['training', 'ecommerce'],
        'connector': {'id': 'none', 'name': 'none', 'type': '.none', 'fields': None},
        'settings': {'syncAlerts': False},
        'owner': 'cases',
    })
    case_id = case_resp['id']
    success(f'Case recreated: id={case_id}')

snap_delete_restore_cycle('snap-case-test', 'Case')

# Delete the case via Cases API
del_resp = req.delete(
    f'{KIBANA_HOST}/api/cases',
    auth=('elastic', KIBANA_PASSWORD),
    headers={'kbn-xsrf': 'true', 'Content-Type': 'application/json'},
    params={'ids': json.dumps([case_id])},
)
del_resp.raise_for_status()
warn(f'Accidentally deleted case {case_id}')

restore_kibana_state(es, SNAP_REPO, 'snap-case-test')
time.sleep(3)

cases_after = kibana_get('/api/cases/_find?perPage=50')
assert any(c['id'] == case_id for c in cases_after.get('cases', [])), 'Case should be restored'
success(f'Case {case_id} successfully restored!')
kibana_link('/app/cases', 'Cases — verify restored case')